# Lab 6: Covert Channels in Image Metadata
**Guided investigation | Student notebook**

**Scenario:** Six images contain pieces of a fictional cybersecurity incident narrative. Each image contains **one partial secret**. Your task is to find the six pieces, document how each was recovered, and reconstruct the full message in the correct order. **Do not execute any activity described by the messages.**

**Learning goals:** 
-  distinguish EXIF, PNG text, and XMP metadata; 
-  use short Python snippets to extract metadata;
-  recognize encoded text; 
-  record reproducible forensic evidence; 
-  assess the limitations of metadata-based conclusions.

**How this lab works:** First practise each small snippet on **one image**. Then select and run the relevant snippets on **each of the six images**, one at a time. You will write the final secret yourself. **No completed secret is provided in this student notebook.**

## 0. Prepare the environment
Keep this notebook beside the `Lab6` folder, which contains `Images`. Run the following two short cells.

In [1]:
%pip -q install Pillow

In [2]:
from pathlib import Path
from PIL import Image, ExifTags

image_dir = Path("Images") if Path("Images").exists() else Path("Lab6/Images")
print("Images found:", len(list(image_dir.glob("*.png"))))


Images found: 6


### Your evidence log
For **each image**, record the metadata type, field name, raw value (or a short excerpt), decoding method (if any), recovered message, and your interpretation. Do not assume that `getexif()` displays all metadata.

## 1. First technique: EXIF metadata (one image)
EXIF metadata can include camera details, dates, descriptions, and other tags. Try **Brain-Channel.png** first.

In [3]:
image_path = image_dir / "Brain-Channel.png"
with Image.open(image_path) as img:
    exif = img.getexif()
    print("EXIF tags:", len(exif))
    for tag_id, value in exif.items():
        print(ExifTags.TAGS.get(tag_id, tag_id), ":", value)

EXIF tags: 5
ResolutionUnit : 2
ImageDescription : LATERAL: Pivot to internal Domain Controller using Mimikatz.
YCbCrPositioning : 1
XResolution : 72.0
YResolution : 72.0


**Observe:** Which EXIF field carries a message? Record it in your evidence log.

**Try it yourself:** Change `image_path` to a different image. If `EXIF tags: 0` appears, does that mean the image has no metadata? Explain.

In [4]:
# Your turn: choose another image and repeat the EXIF check.
image_path = image_dir / "Activation.png"
with Image.open(image_path) as img:
    exif = img.getexif()
    print("EXIF tags:", len(exif))
    for tag_id, value in exif.items():
        print(ExifTags.TAGS.get(tag_id, tag_id), ":", value)
    
    # Check Exif Sub-IFD (tag 34665 / ExifOffset)
    sub_ifd = exif.get_ifd(ExifTags.IFD.Exif)
    print("\nExif Sub-IFD tags:", len(sub_ifd))
    for tag_id, value in sub_ifd.items():
        print(" ", ExifTags.TAGS.get(tag_id, tag_id), ":", value)


EXIF tags: 10
GPSInfo : 396
ResolutionUnit : 2
ExifOffset : 204
Make : Nikon
Model : D750
Software : Adobe Lightroom 13.0
DateTime : 2025:02:02 10:15:00
YCbCrPositioning : 1
XResolution : 72.0
YResolution : 72.0

Exif Sub-IFD tags: 7
  ExifVersion : b'0232'
  ComponentsConfiguration : b'\x01\x02\x03\x00'
  FlashPixVersion : b'0100'
  DateTimeOriginal : 2025:02:02 10:15:00
  DateTimeDigitized : 2025:02:02 10:15:00
  ColorSpace : 65535
  UserComment : b'ASCII\x00\x00\x00VULN: Identify unpatched CVE-2024-XXXX in web gateway.'


## 2. Second technique: PNG text metadata (one image)
PNG files can contain textual information that is **not** returned by `getexif()`. Try **Brain.png**.

In [5]:
image_path = image_dir / "Brain.png"
with Image.open(image_path) as img:
    for key, value in img.info.items():
        if isinstance(value, str):
            print(key, ":", value)

Software : RECON: Perform DNS harvesting and Subdomain enumeration.


**Observe:** Which PNG text field contains a clue? Try **Chemicals.png** next using the same short snippet.

In [6]:
image_path = image_dir / "Chemicals.png"
with Image.open(image_path) as img:
    for key, value in img.info.items():
        if isinstance(value, str):
            print(key, ":", value)


ImageDescription : EXPLOIT: Deploy SQL Injection payload to bypass WAF.


## 3. Third technique: XMP metadata (one image)
XMP is XML-based metadata. It may be stored in PNG files even when EXIF is empty. Try **Dopamine.png**.

In [7]:
image_path = image_dir / "Dopamine.png"
with Image.open(image_path) as img:
    xmp = img.info.get("XML:com.adobe.xmp") or img.info.get("xmp")
    if isinstance(xmp, bytes):
        xmp = xmp.decode("utf-8", errors="replace")
    print(xmp if xmp else "No XMP found")

<?xpacket begin='﻿' id='W5M0MpCehiHzreSzNTczkc9d'?>
<x:xmpmeta xmlns:x='adobe:ns:meta/' x:xmptk='Image::ExifTool 12.76'>
<rdf:RDF xmlns:rdf='http://www.w3.org/1999/02/22-rdf-syntax-ns#'>

 <rdf:Description rdf:about=''
  xmlns:exif='http://ns.adobe.com/exif/1.0/'>
  <exif:MakerNote>PERSIST: Establish reverse shell via obfuscated cron job.</exif:MakerNote>
 </rdf:Description>
</rdf:RDF>
</x:xmpmeta>
<?xpacket end='r'?>


**Observe:** Locate the XML element containing the partial secret. Why did the EXIF-only approach miss it? **Note:** XMP may appear under either `XML:com.adobe.xmp` or `xmp`, depending on how the file was created and the software version.

## 4. Fourth technique: Decode suspicious text (one image)
Sometimes metadata contains **encoded** rather than readable text. Try **Music.png**. First inspect its EXIF or PNG text fields. Identify a value that resembles Base64, then decode it using this small example.

In [8]:
import base64

example = "SGVsbG8="
print(base64.b64decode(example).decode("utf-8"))

Hello


**Your turn:** Replace `example` with the suspicious metadata value from **Music.png**. Do not assume every long string is Base64; use the context and validate the result.

In [9]:
encoded_message = "RVhGSUw6IFR1bm5lbCBjb21wcmVzc2VkIGRhdGFiYXNlIGR1bXAgdmlhIEROUyBDb3ZlcnQgQ2hhbm5lbC4="  # Extracted from Music.png Copyright tag
if encoded_message:
    print(base64.b64decode(encoded_message).decode("utf-8"))


EXFIL: Tunnel compressed database dump via DNS Covert Channel.


## 5. Your investigation: six images, one by one
Use **only the relevant short snippets** from Sections 1–4 for each image. For each image: (1) try EXIF; (2) if the clue is not visible, inspect PNG text; (3) inspect XMP if present; (4) decode only when needed; (5) document the partial secret.

| Image | Metadata type / field | Decoding needed? | Partial secret recovered |
|---|---|---|---|
| Activation.png | EXIF Sub-IFD / `UserComment` (tag 37510) | Yes (strip 8-byte `b'ASCII\x00\x00\x00'` prefix) | VULN: Identify unpatched CVE-2024-XXXX in web gateway. |
| Brain-Channel.png | EXIF IFD0 / `ImageDescription` (tag 270) | None (plain text) | LATERAL: Pivot to internal Domain Controller using Mimikatz. |
| Brain.png | PNG text chunk / `Software` | None (plain text) | RECON: Perform DNS harvesting and Subdomain enumeration. |
| Chemicals.png | PNG text chunk / `ImageDescription` | None (plain text) | EXPLOIT: Deploy SQL Injection payload to bypass WAF. |
| Dopamine.png | XMP packet / `<exif:MakerNote>` in `XML:com.adobe.xmp` | None (XML element parsing) | PERSIST: Establish reverse shell via obfuscated cron job. |
| Music.png | EXIF IFD0 & PNG text / `Copyright` (tag 33432) | Yes (RFC 4648 Base64 decoding) | EXFIL: Tunnel compressed database dump via DNS Covert Channel. |

**Important:** The filename order is **not** necessarily the narrative order.


### Investigation 1: `Activation.png`
Choose and reuse the appropriate short extraction snippet(s) above. Record the evidence in your table. **What is the partial secret?**

In [10]:
image_path = image_dir / "Activation.png"
with Image.open(image_path) as img:
    exif = img.getexif()
    sub_ifd = exif.get_ifd(ExifTags.IFD.Exif)
    raw_comment = sub_ifd.get(37510)
    if raw_comment:
        if raw_comment.startswith(b"ASCII\x00\x00\x00"):
            secret = raw_comment[8:].decode("utf-8")
        else:
            secret = raw_comment.decode("utf-8", errors="ignore")
        print("Found in UserComment (Sub-IFD 37510):", secret)


Found in UserComment (Sub-IFD 37510): VULN: Identify unpatched CVE-2024-XXXX in web gateway.


### Investigation 2: `Brain-Channel.png`
Choose and reuse the appropriate short extraction snippet(s) above. Record the evidence in your table. **What is the partial secret?**

In [11]:
image_path = image_dir / "Brain-Channel.png"
with Image.open(image_path) as img:
    exif = img.getexif()
    secret = exif.get(270)
    print("Found in EXIF ImageDescription (tag 270):", secret)


Found in EXIF ImageDescription (tag 270): LATERAL: Pivot to internal Domain Controller using Mimikatz.


### Investigation 3: `Brain.png`
Choose and reuse the appropriate short extraction snippet(s) above. Record the evidence in your table. **What is the partial secret?**

In [12]:
image_path = image_dir / "Brain.png"
with Image.open(image_path) as img:
    secret = img.info.get("Software")
    print("Found in PNG text chunk 'Software':", secret)


Found in PNG text chunk 'Software': RECON: Perform DNS harvesting and Subdomain enumeration.


### Investigation 4: `Chemicals.png`
Choose and reuse the appropriate short extraction snippet(s) above. Record the evidence in your table. **What is the partial secret?**

In [13]:
image_path = image_dir / "Chemicals.png"
with Image.open(image_path) as img:
    secret = img.info.get("ImageDescription")
    print("Found in PNG text chunk 'ImageDescription':", secret)


Found in PNG text chunk 'ImageDescription': EXPLOIT: Deploy SQL Injection payload to bypass WAF.


### Investigation 5: `Dopamine.png`
Choose and reuse the appropriate short extraction snippet(s) above. Record the evidence in your table. **What is the partial secret?**

In [14]:
import xml.etree.ElementTree as ET

image_path = image_dir / "Dopamine.png"
with Image.open(image_path) as img:
    xmp = img.info.get("XML:com.adobe.xmp") or img.info.get("xmp")
    if isinstance(xmp, bytes):
        xmp = xmp.decode("utf-8", errors="replace")
    clean_xmp = xmp.lstrip("\ufeff")
    root = ET.fromstring(clean_xmp)
    for elem in root.iter():
        if elem.tag.endswith("MakerNote"):
            secret = elem.text
            print("Found in XMP <exif:MakerNote>:", secret)
            break


Found in XMP <exif:MakerNote>: PERSIST: Establish reverse shell via obfuscated cron job.


### Investigation 6: `Music.png`
Choose and reuse the appropriate short extraction snippet(s) above. Record the evidence in your table. **What is the partial secret?**

In [15]:
import base64

image_path = image_dir / "Music.png"
with Image.open(image_path) as img:
    exif = img.getexif()
    b64_val = exif.get(33432) or img.info.get("Copyright")
    secret = base64.b64decode(b64_val).decode("utf-8")
    print("Found in EXIF/PNG Copyright (Base64 decoded):", secret)


Found in EXIF/PNG Copyright (Base64 decoded): EXFIL: Tunnel compressed database dump via DNS Covert Channel.


## 6. Reconstruct the complete secret
Arrange the six recovered fragments into a logical sequence. Use the **stage labels** found in the evidence to justify your order. Do **not** use image filenames as the ordering rule.

1. **Stage:** RECON (from `Brain.png`)  
   **Recovered message:** RECON: Perform DNS harvesting and Subdomain enumeration.
2. **Stage:** VULN (from `Activation.png`)  
   **Recovered message:** VULN: Identify unpatched CVE-2024-XXXX in web gateway.
3. **Stage:** EXPLOIT (from `Chemicals.png`)  
   **Recovered message:** EXPLOIT: Deploy SQL Injection payload to bypass WAF.
4. **Stage:** PERSIST (from `Dopamine.png`)  
   **Recovered message:** PERSIST: Establish reverse shell via obfuscated cron job.
5. **Stage:** LATERAL (from `Brain-Channel.png`)  
   **Recovered message:** LATERAL: Pivot to internal Domain Controller using Mimikatz.
6. **Stage:** EXFIL (from `Music.png`)  
   **Recovered message:** EXFIL: Tunnel compressed database dump via DNS Covert Channel.

**Explain your reasoning:** Why is this the most plausible order?

The reconstructed six-stage narrative maps directly to the standard chronological progression of an Advanced Persistent Threat (APT) attack lifecycle, conforming to both the Lockheed Martin Cyber Kill Chain and the MITRE ATT&CK Enterprise Framework:

1. **Stage 1 — RECON (Reconnaissance / MITRE ATT&CK TA0043):** The threat actor initiates passive and active intelligence gathering via DNS harvesting and subdomain enumeration to map the target organization's external attack surface and identify internet-facing hostnames.
2. **Stage 2 — VULN (Weaponization & Vulnerability Analysis / MITRE ATT&CK TA0001):** Utilizing reconnaissance data, the adversary analyzes the target's external edge infrastructure and discovers an unpatched vulnerability (CVE-2024-XXXX) in the public web application gateway.
3. **Stage 3 — EXPLOIT (Exploitation & Initial Access / MITRE ATT&CK TA0002):** The adversary weaponizes the discovered weakness by deploying a tailored SQL injection payload, bypassing the Web Application Firewall (WAF) to achieve initial arbitrary code/query execution.
4. **Stage 4 — PERSIST (Installation & Persistence / MITRE ATT&CK TA0003):** To prevent losing access if the initial exploit vector is remediated or connections drop, the attacker installs an obfuscated cron job on the compromised host that initiates an automated reverse shell callback to attacker C2 infrastructure.
5. **Stage 5 — LATERAL (Lateral Movement & Credential Access / MITRE ATT&CK TA0008, TA0006):** Operating from their initial foothold, the adversary executes Mimikatz in memory to harvest high-privilege Kerberos tickets/NTLM hashes from LSASS and pivots laterally across internal network segments to compromise the central Domain Controller.
6. **Stage 6 — EXFIL (Exfiltration & Actions on Objectives / MITRE ATT&CK TA0010):** Having attained domain dominance and located confidential database assets, the attacker extracts, compresses, and exfiltrates the sensitive database dump via an encrypted DNS covert channel (port 53 UDP) to evade perimeter egress inspection.

Any alternative permutation is operationally implausible. For example, an attacker cannot pivot laterally (LATERAL) before establishing access (EXPLOIT/PERSIST), nor can they exfiltrate crown-jewel assets (EXFIL) before reconnaissance (RECON) and exploitation have occurred. Furthermore, the alphabetical filename ordering (`Activation.png` -> `Brain-Channel.png` -> `Brain.png` -> `Chemicals.png` -> `Dopamine.png` -> `Music.png`) would sequence VULN -> LATERAL -> RECON -> EXPLOIT -> PERSIST -> EXFIL, which is completely backwards and non-viable. Therefore, the RECON -> VULN -> EXPLOIT -> PERSIST -> LATERAL -> EXFIL sequence is the only logically, chronologically, and forensically sound reconstruction.


## 7. Build an automated metadata scanner
**Learning goal:** Combine the short snippets from Parts 1–4 into a reusable tool, as you did when developing security scanners in Lab 5.

Your scanner must:
1. Loop through **all six images** automatically.
2. Extract **EXIF**, **PNG text**, and **XMP** metadata where available.
3. Record each image's filename, format, metadata field names, and any suspected hidden-message field(s).
4. Identify encoded content and explain any decoding method used; do not automatically assume all unusual text is malicious.
5. Assign a **simple, explainable risk score** and record the reason for each score.
6. Export one row per image to `metadata_report.csv`.

**Suggested classroom risk model:** +10 for a confirmed covert-message field, +5 for a suspicious encoded value requiring investigation, and +5 for an additional unusual metadata indicator supported by evidence. Avoid double-counting an encoded message as both an encoded indicator and an additional unusual indicator. These scores are **heuristics**, not forensic probabilities.

**Step 7A — Starter: loop through the six images.**

In [16]:
for image_path in sorted(image_dir.glob("*.png")):
    with Image.open(image_path) as img:
        print(image_path.name, "→", img.format, "| fields:", len(img.info))


Activation.png → PNG | fields: 8
Brain-Channel.png → PNG | fields: 4
Brain.png → PNG | fields: 4
Chemicals.png → PNG | fields: 1
Dopamine.png → PNG | fields: 5
Music.png → PNG | fields: 8


**Step 7B — Starter: create one result row.** Build on this structure rather than copying a complete solution.

In [17]:
image_path = image_dir / "Brain.png"
with Image.open(image_path) as img:
    row = {
        "filename": image_path.name,
        "format": img.format,
        "metadata_fields": ", ".join(img.info.keys()),
        "risk_score": 0,
    }
print(row)

{'filename': 'Brain.png', 'format': 'PNG', 'metadata_fields': 'srgb, gamma, dpi, Software', 'risk_score': 0}


**Step 7C — Your implementation:** Complete the scanner below or create `metadata_scanner.py`.

Reuse the snippets you have already practised. Extend the result dictionary to include `suspected_field`, `encoding`, and `reason`. The scanner should record the relevant evidence and explain why a risk score was assigned. You may use `csv.DictWriter` from the starter code. **Do not hard-code the six secret messages as the detector's output.**

In [18]:
import csv
import sys
from pathlib import Path

# Ensure metadata_scanner can be imported whether notebook is run from Lab6 or project root
for p in [Path("."), Path("Lab6")]:
    if (p / "metadata_scanner.py").exists() and str(p.resolve()) not in sys.path:
        sys.path.insert(0, str(p.resolve()))

try:
    import metadata_scanner
    output_csv_path = Path("metadata_report.csv") if Path("Images").exists() else Path("Lab6/metadata_report.csv")
    results = metadata_scanner.scan_directory(
        image_dir=image_dir,
        output_csv=output_csv_path,
        verbose=False,
        quiet=False
    )
    print(f"\nSuccessfully saved {len(results)} rows to {output_csv_path}")
except ImportError as e:
    print(f"Error loading metadata_scanner: {e}")


Scanning directory: C:\Users\BillyZ\Desktop\FSCT8561-Labs-Billy-Zandan\Lab6\Images
Found 6 PNG image(s) to inspect.

[1/6] Scanned Activation.png
      Suspected Field : UserComment
      Encoding        : EXIF ASCII header / plain text
      Risk Score      : 15
      Reason          : Confirmed covert message in UserComment (+10); camera hardware and GPS tags embedded in graphic (+5).

[2/6] Scanned Brain-Channel.png
      Suspected Field : ImageDescription
      Encoding        : None (plain text)
      Risk Score      : 10
      Reason          : Confirmed covert message in ImageDescription (+10).

[3/6] Scanned Brain.png
      Suspected Field : Software
      Encoding        : None (plain text)
      Risk Score      : 10
      Reason          : Confirmed covert message in Software (+10).

[4/6] Scanned Chemicals.png
      Suspected Field : ImageDescription
      Encoding        : None (plain text)
      Risk Score      : 10
      Reason          : Confirmed covert message in Image

**Reflection questions**

1. Why can an image contain a hidden message even when `getexif()` returns no tags?
Pillow's `getexif()` method is designed specifically to read standard TIFF/EXIF IFD (Image File Directory) structures (such as IFD0, and optionally Sub-IFDs via `get_ifd()`). However, image containers like PNG support multiple independent metadata storage layers outside the standard EXIF chunk:
- **Native PNG Text Chunks:** PNG files store key-value metadata in dedicated ancillary chunks (`tEXt`, `zTXt`, `iTXt`), which Pillow parses directly into the `img.info` dictionary rather than `getexif()`. In our investigation, `Brain.png` and `Chemicals.png` contained zero EXIF tags (`len(exif) == 0`), yet both carried covert payloads in the native `Software` and `ImageDescription` text chunks.
- **Adobe XMP Packets:** Adobe Extensible Metadata Platform (XMP) embeds metadata as serialized XML/RDF packets inside an `iTXt` chunk with the keyword `XML:com.adobe.xmp` or an application marker. `getexif()` completely ignores XMP packets because they are not organized into numeric EXIF tag tables. In `Dopamine.png`, EXIF was completely absent, but the embedded XMP packet concealed the `PERSIST` secret within an `<exif:MakerNote>` XML element.
- **Sub-IFD Indirection:** Furthermore, even when EXIF data is present, `getexif()` only surfaces root IFD0 tags by default. Sub-IFDs (such as ExifOffset tag 34665 or GPSInfo tag 34853) require explicit parsing via `exif.get_ifd()`. In `Activation.png`, root IFD0 did not contain the message directly; rather, it was nested inside Sub-IFD 34665 (`UserComment`).
Therefore, an empty `getexif()` result merely confirms the absence of root TIFF/EXIF IFD0 structures; it does not indicate that the file is free of metadata.

2. How is **encoding** different from **encryption**? Is Base64 a secure way to hide information?
**Encoding vs. Encryption:**
- **Encoding** is a reversible data representation transformation designed to ensure data integrity, compatibility, or transmission across communication media that cannot handle raw binary bytes (such as converting arbitrary 8-bit binary data into the 64-character ASCII safe subset specified by RFC 4648). Encoding uses publicly standardized algorithms (e.g., Base64, Hexadecimal, ASCII), requires **no secret key**, and can be reversed by anyone who identifies the encoding scheme.
- **Encryption**, by contrast, transforms plaintext into ciphertext using a mathematical algorithm parameterized by a **cryptographic key** (symmetric or asymmetric, such as AES-256-GCM or RSA-4096), with the explicit objective of providing confidentiality. Without the secret key, recovering the underlying plaintext is computationally infeasible.
**Is Base64 secure?**
Base64 provides **zero confidentiality or security**. It is an obfuscation mechanism, not cryptographic encryption. Anyone who inspects the metadata—such as viewing `Music.png`'s `Copyright` field—can immediately recognize the characteristic 64-character alphabet and `=` padding and reverse the string in a single command (`base64.b64decode`). In forensic and threat intelligence contexts, treating Base64 as encryption violates the Kerckhoffs principle and security-by-obscurity tenets.

3. Why should investigators preserve the original files and record hashes?
Preserving pristine original evidence and recording cryptographic hashes (such as SHA-256 and MD5) is foundational to digital forensics and the legal chain of custody:
- **Evidentiary Integrity and Authentication:** Digital evidence is inherently volatile and vulnerable to inadvertent tampering. Opening an image in a graphics viewer or editing software can rewrite ancillary metadata chunks, strip EXIF headers, recompress pixel data, or alter filesystem timestamps. Calculating a cryptographic hash immediately upon evidence acquisition creates a mathematical fingerprint. Any future alteration—even a single bit flip—radically changes the hash value (the avalanche effect).
- **Legal Admissibility (FRE 902 / ISO/IEC 27037):** In judicial proceedings, evidence must satisfy Federal Rules of Evidence Rule 902 (Self-authenticating electronic records) and standards such as ISO/IEC 27037 for digital evidence handling. Demonstrating that the SHA-256 hash of an artifact in court matches the hash recorded at the time of initial seizure proves the evidence was not forged, altered, or contaminated.
- **Forensic Reproducibility:** In our lab audit, the SHA-256 hashes of all six images (e.g., `1730dd61...` for `Activation.png`) were verified prior to and after scanning, ensuring that every finding and subsequent tool run operates on an untampered forensic baseline.

4. Can metadata alone prove that an image was edited or that an incident occurred? What corroborating evidence would you need?
**Can metadata alone prove an incident?**
No. Metadata is circumstantial and highly mutable. Unlike cryptographic signatures, metadata fields can be trivially forged, injected, manipulated, or stripped using readily available tools like `exiftool`, standard Python scripts, or simple hex editors without leaving intrinsic cryptographic traces in the metadata tags themselves. An adversary could intentionally forge camera tags (e.g., embedding Nikon D750 camera details and GPS coordinates in `Activation.png`) to plant false flags or frame third parties.
**Necessary Corroborating Evidence:**
To build a legally defensible and forensically sound case, investigators must corroborate metadata artifacts across multiple orthogonal data sources:
- **Host-Based Forensic Artifacts:** Web server access and error logs (confirming SQL injection attempts matching `Chemicals.png`), crontab configuration files and shell process execution histories (confirming reverse shells matching `Dopamine.png`), and memory dumps / LSASS process handle logs (e.g., Sysmon Event ID 10 confirming Mimikatz activity matching `Brain-Channel.png`).
- **Network Traffic & Telemetry:** Passive DNS query logs and packet captures (PCAP) validating subdomain enumeration (`Brain.png`) and outbound DNS tunneling / high-volume TXT query exfiltration (`Music.png`).
- **Endpoint Detection and Response (EDR) & SIEM Alerts:** Corroborated timestamps, file system journal entries (USN Journal, $MFT timestamps), and perimeter firewall / WAF connection logs confirming the external IP addresses involved in the breach.

5. How does this investigation connect to the automated evidence collection and analysis you practised in Labs 4 and 5?
This investigation represents the natural evolution and culmination of the forensic automation methodologies developed in Labs 4 and 5:
- **Automated Evidence Collection (Lab 4):** In Lab 4, we built automated Python pipelines to sweep networks, collect volatile host metrics, and aggregate security logs across multiple distributed systems into structured tabular formats. In Lab 6, this programmatic paradigm was applied to image media carriers: rather than manually clicking through image properties in a desktop GUI, our automated scanner systematically parsed multi-tier metadata structures (EXIF, PNG text, XMP) across entire directories within seconds.
- **Security Scanners & Rule-Based Detection (Lab 5):** In Lab 5, we engineered defensive vulnerability scanners that evaluated target attributes against signature rules, handled protocol corner cases, and assigned prioritized risk metrics to security findings. In Lab 6, we designed a similar heuristic detection and scoring engine (+10 confirmed covert message, +5 encoding/obfuscation, +5 anomalous container/indicator) that prevents double-counting and exports findings into standardized RFC 4180 CSV reports (`metadata_report.csv`).
- **Forensic Lifecycle Integration:** Both domains reinforce the core DFIR principle that security operations cannot scale with manual inspection alone. Scripted automation ensures reproducibility, eliminates human oversight across multi-layered data containers, and enforces strict forensic integrity from raw collection to structured threat intelligence.


## Submission requirements
Submit **one PDF** named `Lab6-FirstName-LastName-StudentNumber.pdf` that includes:

- **Metadata evidence table** for all six images, identifying the metadata type, field, relevant raw value, decoding method (if applicable), and recovered fragment.
- **Screenshots of the extraction output** showing how each partial secret was found.
- **Complete reconstructed six-stage secret** in the correct order, with a brief explanation of the sequence.
- **Your completed Python metadata scanner** (code and brief explanation of its main decisions).
- **Scanner results:** `metadata_report.csv` output as a table, including the risk score and supporting reason for each image.
- **Brief interpretation:** Explain one limitation of EXIF-only extraction and one limitation of automated risk scoring.
- **Answer to reflection questions** 

Do not execute the activities described by the recovered messages.

## Grading rubric (100 points)

| Component | Points | Full-credit expectations |
|---|---:|---|
| Metadata extraction and evidence documentation | **25** | Appropriate EXIF/PNG/XMP extraction for all six images; accurate field identification; clear evidence screenshots and tables. |
| Covert-channel detection and decoding | **25** | All six hidden fragments recovered accurately; encoded content decoded correctly; extraction methods explained. |
| Six-stage secret reconstruction | **20** | Complete secret assembled in correct order with a logical justification. |
| Automated metadata scanner and CSV report | **20** | Scanner iterates through all six images, checks relevant metadata types, identifies suspected fields, exports complete CSV rows, and applies consistent, explainable risk scores. |
| Interpretations, reflections, code quality, and presentation | **10** | Readable, reproducible Python code; clear report organization; thoughtful explanation of EXIF-only and risk-scoring limitations. |
| **Total** | **100** | |